# Lab 6: Language Identification with FastText Word Vectors

**Dataset**: IndicCorpV2 (24 Indic languages, 1000 sentences each) from Lab 1  
**Features**: FastText pretrained wiki word vectors (300-dim)  
**Metric**: Custom Macro-F1 — no sklearn used  

| # | Classifier | Feature Type |
|---|---|---|
| 1 | Logistic Regression | Averaged word vectors |
| 2 | MLP n=0 (linear) | Averaged word vectors |
| 3 | MLP n=1 | Averaged word vectors |
| 4 | MLP n=2 | Averaged word vectors |
| 5 | LSTM | Sequence of word vectors |
| 6 | GRU | Sequence of word vectors |
| 7 | FastText Classification | Raw text (fasttext library) |

### Step 0 — Install dependencies (run once)

In [1]:
# Uncomment and run once if needed
# !pip install torch fasttext numpy


### Step 1 — Data Loading & Stratified Split

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from lab6_solution import (
    load_data, stratified_split, LabelEncoder, RANDOM_SEED
)

data = load_data()
train_data, val_data, test_data = stratified_split(data)
print(f'Total: {len(data)} | Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

from collections import Counter
lbl_counts = Counter(l for _, l in data)
print(f'Languages: {len(lbl_counts)}  |  Min per lang: {min(lbl_counts.values())}  |  Max: {max(lbl_counts.values())}')


Total: 20000 | Train: 16000 | Val: 2000 | Test: 2000
Languages: 20  |  Min per lang: 1000  |  Max: 1000


### Step 2 — FastText Wiki Word Vectors

In [3]:
from lab6_solution import load_fasttext_vecs, LANG_TO_FT

print('Language → FastText code mapping:')
for lang, ft in sorted(LANG_TO_FT.items()):
    print(f'  {lang:<15} → wiki.{ft}.vec')

print('\nLoading vectors (downloads if not cached)...')
vecs = load_fasttext_vecs(train_data + val_data + test_data)
print(f'Total word vectors loaded: {len(vecs):,}')


Language → FastText code mapping:
  asm_Beng        → wiki.as.vec
  ben_Beng        → wiki.bn.vec
  brx_Deva        → wiki.hi.vec
  doi_Deva        → wiki.hi.vec
  gom_Deva        → wiki.hi.vec
  guj_Gujr        → wiki.gu.vec
  hin_Deva        → wiki.hi.vec
  kan_Knda        → wiki.kn.vec
  kas_Arab        → wiki.ur.vec
  kas_Deva        → wiki.hi.vec
  mai_Deva        → wiki.hi.vec
  mal_Mlym        → wiki.ml.vec
  mar_Deva        → wiki.mr.vec
  mni_Beng        → wiki.bn.vec
  mni_Mtei        → wiki.hi.vec
  npi_Deva        → wiki.ne.vec
  ory_Orya        → wiki.or.vec
  pan_Guru        → wiki.pa.vec
  san_Deva        → wiki.sa.vec
  sat_Olck        → wiki.hi.vec
  snd_Arab        → wiki.sd.vec
  tam_Taml        → wiki.ta.vec
  tel_Telu        → wiki.te.vec
  urd_Arab        → wiki.ur.vec

Loading vectors (downloads if not cached)...
[Vecs] Loading cached word vectors from c:\Users\krish\Desktop\LLMS\LAB6(3-9-26)\fasttext_vecs.npz
Total word vectors loaded: 58,896


### Step 3 — Feature Extraction

In [4]:
import numpy as np
from lab6_solution import (
    make_avg_features, make_seq_features, LabelEncoder, macro_f1
)

le = LabelEncoder().fit([l for _, l in data])
n_classes = len(le.classes_)
print(f'Number of classes: {n_classes}')

y_train = le.transform([l for _, l in train_data])
y_val   = le.transform([l for _, l in val_data])
y_test  = le.transform([l for _, l in test_data])
true_test = [l for _, l in test_data]

# Averaged vectors
X_tr_avg, _ = make_avg_features(train_data, vecs)
X_va_avg, _ = make_avg_features(val_data,   vecs)
X_te_avg, _ = make_avg_features(test_data,  vecs)
print(f'Averaged feature shape: {X_tr_avg.shape}')

# Sequence features
X_tr_seq, _, L_tr = make_seq_features(train_data, vecs)
X_va_seq, _, L_va = make_seq_features(val_data,   vecs)
X_te_seq, _, L_te = make_seq_features(test_data,  vecs)
print(f'Sequence feature shape: {X_tr_seq.shape}')


Number of classes: 20
Averaged feature shape: (16000, 300)
Sequence feature shape: (16000, 50, 300)


### Step 4 — Classifier 1: Logistic Regression (Averaged Word Vectors)

In [5]:
from lab6_solution import LogisticRegression

print('Training Logistic Regression (pure numpy, softmax + SGD)...')
lr_clf = LogisticRegression(n_classes=n_classes, dim=300, lr=0.2, epochs=30)
lr_clf.fit(X_tr_avg, y_train)

preds_lr = le.inverse_transform(lr_clf.predict(X_te_avg))
f1_lr = macro_f1(true_test, preds_lr)
print(f'\nLogistic Regression — Test Macro-F1: {f1_lr:.4f}')


Training Logistic Regression (pure numpy, softmax + SGD)...
    Epoch  10  train-acc=0.8444
    Epoch  20  train-acc=0.8569
    Epoch  30  train-acc=0.8619

Logistic Regression — Test Macro-F1: 0.8501


### Step 5 — Classifiers 2–4: MLP (n=0, 1, 2 hidden layers)

In [6]:
from lab6_solution import build_mlp, _train_torch_model, _predict_torch

mlp_results = {}
for n_hidden in [0, 1, 2]:
    print(f'\n--- MLP n={n_hidden} ---')
    model = build_mlp(n_hidden, 300, n_classes)
    _train_torch_model(model, X_tr_avg, y_train, X_va_avg, y_val,
                       le, epochs=20, batch=128, lr=1e-3, seq=False)
    raw = _predict_torch(model, X_te_avg)
    preds = le.inverse_transform(raw)
    f1 = macro_f1(true_test, preds)
    mlp_results[f'MLP n={n_hidden}'] = f1
    print(f'MLP n={n_hidden} — Test Macro-F1: {f1:.4f}')



--- MLP n=0 ---
    Epoch   5  loss=0.0055  val-F1=0.8404
    Epoch  10  loss=0.0043  val-F1=0.8547
    Epoch  15  loss=0.0037  val-F1=0.8572
    Epoch  20  loss=0.0034  val-F1=0.8612
MLP n=0 — Test Macro-F1: 0.8534

--- MLP n=1 ---
    Epoch   5  loss=0.0030  val-F1=0.8687
    Epoch  10  loss=0.0025  val-F1=0.8783
    Epoch  15  loss=0.0022  val-F1=0.8796
    Epoch  20  loss=0.0020  val-F1=0.8798
MLP n=1 — Test Macro-F1: 0.8690

--- MLP n=2 ---
    Epoch   5  loss=0.0030  val-F1=0.8719
    Epoch  10  loss=0.0025  val-F1=0.8796
    Epoch  15  loss=0.0021  val-F1=0.8770
    Epoch  20  loss=0.0019  val-F1=0.8791
MLP n=2 — Test Macro-F1: 0.8754


### Step 6 — Classifier 5: LSTM

In [7]:
from lab6_solution import LSTMClassifier

print('Training LSTM...')
lstm_model = LSTMClassifier.build(300, n_classes, hidden=128)
_train_torch_model(lstm_model,
                   (X_tr_seq, L_tr), y_train,
                   (X_va_seq, L_va), y_val,
                   le, epochs=20, batch=64, lr=1e-3, seq=True)
raw = _predict_torch(lstm_model, X_te_seq, L_te)
preds_lstm = le.inverse_transform(raw)
f1_lstm = macro_f1(true_test, preds_lstm)
print(f'\nLSTM — Test Macro-F1: {f1_lstm:.4f}')


Training LSTM...
    Epoch   5  loss=0.0045  val-F1=0.8813
    Epoch  10  loss=0.0031  val-F1=0.8909
    Epoch  15  loss=0.0023  val-F1=0.8873
    Epoch  20  loss=0.0020  val-F1=0.8979

LSTM — Test Macro-F1: 0.8818


### Step 7 — Classifier 6: GRU

In [8]:
from lab6_solution import GRUClassifier

print('Training GRU...')
gru_model = GRUClassifier.build(300, n_classes, hidden=128)
_train_torch_model(gru_model,
                   (X_tr_seq, L_tr), y_train,
                   (X_va_seq, L_va), y_val,
                   le, epochs=20, batch=64, lr=1e-3, seq=True)
raw = _predict_torch(gru_model, X_te_seq, L_te)
preds_gru = le.inverse_transform(raw)
f1_gru = macro_f1(true_test, preds_gru)
print(f'\nGRU — Test Macro-F1: {f1_gru:.4f}')


Training GRU...
    Epoch   5  loss=0.0039  val-F1=0.8955
    Epoch  10  loss=0.0025  val-F1=0.8966
    Epoch  15  loss=0.0019  val-F1=0.8921
    Epoch  20  loss=0.0018  val-F1=0.8908

GRU — Test Macro-F1: 0.8815


### Step 8 — Classifier 7: FastText Classification

In [9]:
from lab6_solution import run_fasttext_classifier

print('Training FastText classifier...')
ft_preds = run_fasttext_classifier(train_data + val_data, test_data)
if ft_preds is not None:
    f1_ft = macro_f1(true_test, ft_preds)
    print(f'\nFastText Classification — Test Macro-F1: {f1_ft:.4f}')


Training FastText classifier...
fasttext not installed. Run: pip install fasttext


### Step 9 — Final Comparison Table

In [10]:
import pandas as pd

all_results = {
    'Logistic Regression': f1_lr,
    **mlp_results,
    'LSTM': f1_lstm,
    'GRU': f1_gru,
    'FastText Classification': f1_ft if ft_preds is not None else float('nan'),
}

df = pd.DataFrame(list(all_results.items()), columns=['Classifier', 'Macro-F1'])
df = df.sort_values('Macro-F1', ascending=False).reset_index(drop=True)
df.index += 1
display(df.style.format({'Macro-F1': '{:.4f}'}).background_gradient(cmap='RdYlGn', subset=['Macro-F1']))


,Classifier,Macro-F1
1,LSTM,0.8818
2,GRU,0.8815
3,MLP n=2,0.8754
4,MLP n=1,0.8690
5,MLP n=0,0.8534
6,Logistic Regression,0.8501
7,FastText Classification,nan


### Analysis

**Feature representation:**
- Averaged word vectors give a fixed-size sentence representation (300-dim). Simple but loses word order.
- Sequence features (LSTM/GRU) preserve word order, letting the model learn ordering patterns per language.

**Expected ranking:**
1. **FastText Classification** — purpose-built for text classification, uses sub-word n-grams, typically best.
2. **GRU / LSTM** — capture sequential dependencies, better than averaging for morphologically rich languages.
3. **MLP n=2 > n=1 > n=0** — more capacity helps with averaged features.
4. **Logistic Regression** — strong baseline; Indic scripts are highly distinctive so even simple models do well.

**Why Indic LID is generally easy:**  
Indic languages use distinct Unicode scripts (Devanagari, Tamil, Telugu, etc.), so even character-level models achieve high accuracy. The challenge arises between same-script languages (e.g., Hindi/Marathi/Nepali all use Devanagari).

**Macro-F1 vs Accuracy:**  
Macro-F1 treats all 24 classes equally regardless of sample size — ideal for balanced multiclass evaluation.